# Phase 2 — Data Cleaning & Spark Processing
**Project:** Sentiment Analysis of Yelp Reviews at Scale (Apache Spark + NLP)

**Goal of this notebook:** take the clean data from Phase 1 and show how we use **Apache Spark** to inspect it, handle missing values, transform it, aggregate it, join it, and query it with **Spark SQL**. At the end we build the **text preprocessing pipeline** (NLP) that Phase 3 will use for machine learning, and we export small summary tables for the Phase 4 dashboard.

**Input (from Phase 1, on Google Drive):**
- `yelp_project/reviews_clean.parquet`: 6,298,179 reviews (`review_id, business_id, stars, text, date, label, year`)
- `yelp_project/business.parquet`: 150,346 businesses (`business_id, name, city, state, categories`)

**Output (on Google Drive):**
- `yelp_project/models/preprocessing_pipeline`: the fitted Spark ML text pipeline
- `yelp_project/dashboard_data/`: `yearly_stats`, `city_stats`, `category_stats`, `length_stats`

**Rules we follow in this notebook**

| Rule | Why |
|---|---|
| Aggregations, joins and SQL run on the **full** data | Spark spreads this work across all CPU cores, so it scales to millions of rows |
| The text pipeline is fitted and inspected on a **5% sample** | Text processing is the most expensive step; a sample is enough to check that it works |
| No `toPandas()` / `collect()` on the full data | That would copy millions of rows into one Python process and crash the runtime |
| `cache()` only DataFrames we reuse, `unpersist()` when done | Memory on Colab is limited |

> Run on Google Colab with a **CPU + High-RAM** runtime. Run the cells **from top to bottom**.

## 0. Setup

### 0.1 Install PySpark and mount Google Drive
**What:** install the PySpark library and connect Colab to Google Drive.
**Why:** our data lives on Google Drive (it is several GB, so it is not in the GitHub repo). After mounting, Drive looks like a normal folder at `/content/drive/MyDrive`.

In [1]:
!pip install -q pyspark

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 0.2 Configuration (paths)
**What:** all file paths are defined in one place.
**Why:** if the shared folder is somewhere else on your Drive, you only change `BASE_PATH`. The `assert` stops early with a clear message if the Phase 1 files are missing.

In [2]:
import os

BASE_PATH      = "/content/drive/MyDrive/yelp_project"     # shared project folder on Drive
REVIEWS_PATH   = f"{BASE_PATH}/reviews_clean.parquet"       # output of Phase 1
BUSINESS_PATH  = f"{BASE_PATH}/business.parquet"            # output of Phase 1
PIPELINE_PATH  = f"{BASE_PATH}/models/preprocessing_pipeline"
DASHBOARD_PATH = f"{BASE_PATH}/dashboard_data"

for p in [REVIEWS_PATH, BUSINESS_PATH]:
    assert os.path.exists(p), f"Not found: {p}. Run 01_data_preparation.ipynb or add the shared folder shortcut to My Drive."
print("Input files found.")

Input files found.


### 0.3 Start Spark
**What:** create a `SparkSession`, the entry point to everything in Spark.
**Why these settings:**
- `spark.driver.memory = 10g`: on Colab, Spark runs in *local mode* (driver and workers share one machine), so the driver memory is the memory Spark can use.
- `spark.sql.shuffle.partitions = 64`: after a `groupBy` or `join`, Spark splits data into this many pieces. The default (200) creates many tiny tasks on a single machine; 64 is a better fit for Colab.

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("YelpSentiment-Phase2")
         .config("spark.driver.memory", "10g")
         .config("spark.sql.shuffle.partitions", "64")
         .getOrCreate())

print("Spark version:", spark.version)

Spark version: 4.0.5


### 0.4 Load the Parquet files
**What:** read both tables saved in Phase 1 and count their rows.
**Why Parquet:** it is a *columnar* format with the schema stored inside the file. Spark reads only the columns a query needs (e.g. it can count stars without reading the review text), which is much faster than JSON or CSV.

In [4]:
df       = spark.read.parquet(REVIEWS_PATH)
business = spark.read.parquet(BUSINESS_PATH)

total_reviews  = df.count()
total_business = business.count()
print(f"Reviews   : {total_reviews:,}")
print(f"Businesses: {total_business:,}")

Reviews   : 6,298,179
Businesses: 150,346


### 0.5 A 5% sample (for the text pipeline only)
**What:** a random 5% of the reviews (about 315K rows). `seed=42` makes it the same sample for every teammate, and `cache()` keeps it in memory because we reuse it in Section 6.

**When to use `sample` vs the full `df`:**
- **Full data**: counts, averages, `groupBy`, joins, Spark SQL. Spark handles these efficiently, and the numbers we report must come from all reviews.
- **Sample**: heavy text processing while developing and checking (cleaning, tokenizing, statistics on tokens), and anything converted to pandas for plotting (Phase 3).

In [5]:
sample = df.sample(fraction=0.05, seed=42).cache()
total_sample = sample.count()          # the first action fills the cache
print(f"Sample rows: {total_sample:,} ({total_sample / total_reviews:.1%} of all reviews)")

Sample rows: 314,338 (5.0% of all reviews)


## 1. Data quality inspection & missing values
Before analysing data we check that it is complete and consistent: data types, missing values (nulls), empty text, duplicates, and impossible values. Phase 1 already removed some problems from the reviews, so here we **verify** that, and we clean the business table.

### 1.1 Schemas
**What:** print column names and data types of both tables.
**Why:** wrong types cause silent bugs (e.g. a date stored as text cannot be used to extract the month).

In [6]:
print("reviews:")
df.printSchema()
print("business:")
business.printSchema()

reviews:
root
 |-- review_id: string (nullable = true)
 |-- business_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- text: string (nullable = true)
 |-- date: date (nullable = true)
 |-- label: integer (nullable = true)
 |-- year: integer (nullable = true)

business:
root
 |-- business_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- categories: string (nullable = true)



### 1.2 Missing values (nulls) per column
**What:** count the nulls in every column of both tables.
**Why one expression:** we build one `sum(isNull)` expression per column and put them all in **one** `select`. Spark then scans the table **once**. Calling `.count()` in a loop for each column would scan the whole table once per column.

In [7]:
def null_counts(sdf):
    # One row with the number of nulls in every column, computed in ONE pass over the data
    return sdf.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in sdf.columns])

print("Nulls in reviews:")
null_counts(df).show()

print("Nulls in business:")
null_counts(business).show()

Nulls in reviews:
+---------+-----------+-----+----+----+-----+----+
|review_id|business_id|stars|text|date|label|year|
+---------+-----------+-----+----+----+-----+----+
|        0|          0|    0|   0|   0|    0|   0|
+---------+-----------+-----+----+----+-----+----+

Nulls in business:
+-----------+----+----+-----+----------+
|business_id|name|city|state|categories|
+-----------+----+----+-----+----------+
|          0|   0|   0|    0|       103|
+-----------+----+----+-----+----------+



### 1.3 Empty strings
**What:** count text values that are not null but empty (or only spaces).
**Why:** `isNull()` does not catch `""`. An empty category or city is still missing information.

In [8]:
def is_empty(c):
    return (F.trim(F.col(c)) == "").cast("int")

df.select(F.sum(is_empty("text")).alias("empty_text")).show()

business.select(
    F.sum(is_empty("categories")).alias("empty_categories"),
    F.sum(is_empty("city")).alias("empty_city"),
    F.sum(is_empty("name")).alias("empty_name"),
).show()

+----------+
|empty_text|
+----------+
|         0|
+----------+

+----------------+----------+----------+
|empty_categories|empty_city|empty_name|
+----------------+----------+----------+
|               0|         0|         0|
+----------------+----------+----------+



### 1.4 Duplicates
**What:** compare the number of rows with the number of *distinct* IDs.
**Why:** each review and each business must appear once. A duplicated business would double-count reviews after the join in Section 4.

In [9]:
(df.agg(F.count("*").alias("rows"),
        F.countDistinct("review_id").alias("distinct_review_id"))
   .withColumn("duplicates", F.col("rows") - F.col("distinct_review_id"))
   .show())

(business.agg(F.count("*").alias("rows"),
              F.countDistinct("business_id").alias("distinct_business_id"))
   .withColumn("duplicates", F.col("rows") - F.col("distinct_business_id"))
   .show())

+-------+------------------+----------+
|   rows|distinct_review_id|duplicates|
+-------+------------------+----------+
|6298179|           6298179|         0|
+-------+------------------+----------+

+------+--------------------+----------+
|  rows|distinct_business_id|duplicates|
+------+--------------------+----------+
|150346|              150346|         0|
+------+--------------------+----------+



### 1.5 Consistency checks
**What:** look for values that are present but wrong: stars other than 1, 2, 4, 5 (3 was removed in Phase 1), a `label` that does not match the stars, and the date / year range.
**Why:** the label is our ML target. If it were wrong, every later result would be wrong.

In [10]:
df.agg(
    F.sum((~F.col("stars").isin(1.0, 2.0, 4.0, 5.0)).cast("int")).alias("unexpected_stars"),
    F.sum((F.col("label") != F.when(F.col("stars") >= 4, 1).otherwise(0)).cast("int")).alias("label_mismatch"),
    F.min("date").alias("first_date"),
    F.max("date").alias("last_date"),
    F.min("year").alias("min_year"),
    F.max("year").alias("max_year"),
).show()

+----------------+--------------+----------+----------+--------+--------+
|unexpected_stars|label_mismatch|first_date| last_date|min_year|max_year|
+----------------+--------------+----------+----------+--------+--------+
|               0|             0|2005-02-16|2022-01-19|    2005|    2022|
+----------------+--------------+----------+----------+--------+--------+



### 1.6 Handle missing values in `business`
**What:** businesses with a null **or** empty `categories` get the value `"Unknown"`. We count how many were filled.
**Why fill instead of drop:** dropping these businesses would also drop all their reviews when we join the tables. `"Unknown"` keeps the reviews and makes the missing information visible. (`fillna()` only replaces nulls, so we use `when()` to also catch empty strings.)
**Reviews:** the review table needs no filling. Phase 1 already dropped reviews with missing text or stars, and the checks above confirm it.

In [11]:
missing_cat = F.col("categories").isNull() | (F.trim(F.col("categories")) == "")

n_filled = business.filter(missing_cat).count()

business_clean = business.withColumn(
    "categories",
    F.when(missing_cat, F.lit("Unknown")).otherwise(F.col("categories")))

n_still_missing = business_clean.filter(missing_cat).count()
print(f"Businesses with missing categories filled with 'Unknown': {n_filled:,}")
print(f"Missing categories after filling: {n_still_missing}")

Businesses with missing categories filled with 'Unknown': 103
Missing categories after filling: 0


### 1.7 Summary of data-quality findings
*(Numbers from the run on Colab.)*

| Check | Reviews | Business | Action taken |
|---|---|---|---|
| Null values | 0 in all 7 columns | 103 (all in `categories`) | business: filled with `"Unknown"` (1.6) |
| Empty strings | 0 empty text | 0 categories, 0 city, 0 name | none needed |
| Duplicate IDs | 0 | 0 | none needed |
| Unexpected stars / label mismatch | 0 / 0 | – | none needed |
| Date range | 2005-02-16 → 2022-01-19 | – | 2022 is partial (only up to 19 Jan) |

## 2. Spark DataFrame transformations
A **transformation** (`withColumn`, `select`, `filter`, ...) only describes a new DataFrame; Spark does not compute anything yet (*lazy evaluation*). Work happens only at an **action** (`count`, `show`, `write`). This lets Spark optimise the whole chain before running it.

### 2.1 New columns with `withColumn`
**What:** add simple features to each review:
- `n_chars`: number of characters; `n_words`: number of words (split on spaces)
- `month`, `day_of_week` (1 = Sunday ... 7 = Saturday) and `day_name` from the date

**Why:** review length is a strong hint of sentiment (unhappy customers write more), and the date parts let us look for time patterns.

In [12]:
reviews_feat = (df
    .withColumn("n_chars", F.length("text"))
    .withColumn("n_words", F.size(F.split(F.col("text"), r"\s+")))
    .withColumn("month", F.month("date"))
    .withColumn("day_of_week", F.dayofweek("date"))
    .withColumn("day_name", F.date_format("date", "E")))

reviews_feat.select("stars", "label", "date", "month", "day_of_week", "day_name", "n_chars", "n_words").show(5)

+-----+-----+----------+-----+-----------+--------+-------+-------+
|stars|label|      date|month|day_of_week|day_name|n_chars|n_words|
+-----+-----+----------+-----+-----------+--------+-------+-------+
|  4.0|    1|2019-04-07|    4|          1|     Sun|    246|     45|
|  4.0|    1|2019-04-20|    4|          7|     Sat|    112|     22|
|  5.0|    1|2018-10-01|   10|          2|     Mon|    223|     43|
|  2.0|    0|2021-04-18|    4|          1|     Sun|    339|     63|
|  5.0|    1|2015-07-24|    7|          6|     Fri|    328|     66|
+-----+-----+----------+-----+-----------+--------+-------+-------+
only showing top 5 rows


### 2.2 `select` / `drop`: a slim table without the text
**What:** a version of the reviews **without** the `text` column, cached in memory.
**Why:** the text is about 95% of the data size, but Sections 3–5 only need IDs, stars, label, dates and lengths. The slim table is small enough to cache, so all the following aggregations, joins and SQL queries read it from memory instead of re-reading Parquet from Drive.

In [13]:
reviews_slim = reviews_feat.drop("text").cache()
print(f"reviews_slim rows: {reviews_slim.count():,}")   # fills the cache
reviews_slim.printSchema()

reviews_slim rows: 6,298,179
root
 |-- review_id: string (nullable = true)
 |-- business_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- date: date (nullable = true)
 |-- label: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- n_chars: integer (nullable = true)
 |-- n_words: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- day_name: string (nullable = true)



### 2.3 Filtering examples
**What:** `filter()` keeps only the rows that match a condition. Three examples:
1. **Long reviews** (more than 500 words): are they more negative than average?
2. **Year range**: compare the last complete years before COVID-19 (2015–2019) with 2020 and 2021, using `between()` and `when()`.
3. **Filtering on the text itself**: reviews that mention "refund". This one needs the `text` column, so it reads the full table.

**Why:** filtering is how we select subsets for analysis. Spark pushes filters down to the Parquet reader, so rows that do not match are skipped early.

**Example 1, long reviews:** `filter` on the `n_words` column we created, then one `agg` computes both the count and the positive share in a single pass.

In [14]:
# 1. Long reviews
r = (reviews_slim.filter(F.col("n_words") > 500)
     .agg(F.count("*").alias("n"), F.avg("label").alias("pos_share"))
     .first())
print(f"Reviews with > 500 words: {r['n']:,} ({r['n'] / total_reviews:.2%} of all)")
print(f"Positive share among them: {r['pos_share']:.1%}   (all reviews: 74.4%)")

Reviews with > 500 words: 56,681 (0.90% of all)
Positive share among them: 41.6%   (all reviews: 74.4%)


**Example 2, year range:** `between(2015, 2021)` keeps 7 years; `when()` labels each row with its period, and we divide by the number of years so periods of different length can be compared fairly.

In [15]:
# 2. Year range: pre-COVID vs COVID vs recovery
period = (F.when(F.col("year").between(2015, 2019), "a) 2015-2019 (pre-COVID)")
           .when(F.col("year") == 2020, "b) 2020 (COVID-19)")
           .when(F.col("year") == 2021, "c) 2021 (recovery)"))

(reviews_slim
    .filter(F.col("year").between(2015, 2021))
    .withColumn("period", period)
    .groupBy("period")
    .agg(F.count("*").alias("n_reviews"),
         F.countDistinct("year").alias("n_years"),
         F.round(F.avg("label"), 4).alias("positive_share"))
    .withColumn("reviews_per_year", F.round(F.col("n_reviews") / F.col("n_years")).cast("long"))
    .orderBy("period")
    .show(truncate=False))

+------------------------+---------+-------+--------------+----------------+
|period                  |n_reviews|n_years|positive_share|reviews_per_year|
+------------------------+---------+-------+--------------+----------------+
|a) 2015-2019 (pre-COVID)|3709741  |5      |0.7427        |741948          |
|b) 2020 (COVID-19)      |517544   |1      |0.7287        |517544          |
|c) 2021 (recovery)      |574499   |1      |0.7059        |574499          |
+------------------------+---------+-------+--------------+----------------+



**Example 3, filter on the text:** `lower(text).contains("refund")` finds reviews that talk about refunds. This reads the 5 GB text column, so it takes longer than the examples above.

In [16]:
# 3. Filter on the review text (reads the text column of the full data)
r = (reviews_feat.filter(F.lower(F.col("text")).contains("refund"))
     .agg(F.count("*").alias("n"), F.avg("label").alias("pos_share"))
     .first())
print(f"Reviews mentioning 'refund': {r['n']:,}  |  positive share: {r['pos_share']:.1%}")

Reviews mentioning 'refund': 49,284  |  positive share: 8.8%


## 3. Aggregations (`groupBy` + `agg`)
`groupBy` collects rows with the same key; `agg` computes statistics for each group. Spark first computes partial results on each partition, then **shuffles** only those small partial results to combine them. That is why aggregating millions of rows is fast. All aggregations here use the **full** data (`reviews_slim`).

### 3.1 Reviews and positive share per year
**What:** number of reviews, share of positive reviews (`avg(label)`, because label is 0/1), average stars and average length per year.
**Why:** this shows growth of the platform, the COVID-19 drop and how sentiment changes over time. We keep this table (`yearly_df`) for the SQL comparison (Section 5) and the dashboard export (Section 7).

In [17]:
yearly_df = (reviews_slim.groupBy("year")
    .agg(F.count("*").alias("n_reviews"),
         F.round(F.avg("label"), 4).alias("positive_share"),
         F.round(F.avg("stars"), 3).alias("avg_stars"),
         F.round(F.avg("n_words"), 1).alias("avg_words"))
    .orderBy("year"))

yearly_df.show(20)

+----+---------+--------------+---------+---------+
|year|n_reviews|positive_share|avg_stars|avg_words|
+----+---------+--------------+---------+---------+
|2005|      683|         0.877|    4.126|     71.6|
|2006|     3201|        0.8903|    4.198|     98.4|
|2007|    12458|        0.8511|    4.066|    120.1|
|2008|    39220|        0.8384|    3.982|    138.8|
|2009|    61614|        0.7952|    3.869|    137.3|
|2010|   115549|        0.7938|    3.865|    134.0|
|2011|   194809|        0.7802|    3.835|    129.2|
|2012|   245187|        0.7621|    3.793|    125.5|
|2013|   332536|        0.7552|    3.789|    119.8|
|2014|   461656|        0.7535|    3.818|    110.9|
|2015|   616159|        0.7486|    3.829|    103.5|
|2016|   686637|        0.7466|    3.846|    100.4|
|2017|   744886|        0.7461|    3.857|     98.0|
|2018|   829238|        0.7414|    3.854|     96.4|
|2019|   832821|        0.7335|    3.835|     94.7|
|2020|   517544|        0.7287|    3.839|     93.2|
|2021|   574

### 3.2 Reviews per star rating
**What:** count and percentage of all reviews for each star value, with its label and average length.
**Why:** it shows how the 0/1 labels are built from the stars and which star values dominate.

In [18]:
(reviews_slim.groupBy("stars", "label")
    .agg(F.count("*").alias("n_reviews"),
         F.round(F.avg("n_words"), 1).alias("avg_words"))
    .withColumn("pct_of_all", F.round(F.col("n_reviews") / total_reviews * 100, 2))
    .orderBy("stars")
    .show())

+-----+-----+---------+---------+----------+
|stars|label|n_reviews|avg_words|pct_of_all|
+-----+-----+---------+---------+----------+
|  1.0|    0|  1069507|    134.0|     16.98|
|  2.0|    0|   544228|    134.3|      8.64|
|  4.0|    1|  1452876|    108.0|     23.07|
|  5.0|    1|  3231568|     84.5|     51.31|
+-----+-----+---------+---------+----------+



### 3.3 Reviews per label (class balance)
**What:** number and percentage of positive and negative reviews.
**Why:** the classes are imbalanced (about 3 : 1). Phase 3 must use class weights and report F1 / Recall, not only accuracy.

In [19]:
(reviews_slim.groupBy("label")
    .agg(F.count("*").alias("n_reviews"))
    .withColumn("sentiment", F.when(F.col("label") == 1, "positive").otherwise("negative"))
    .withColumn("pct_of_all", F.round(F.col("n_reviews") / total_reviews * 100, 2))
    .orderBy("label")
    .show())

+-----+---------+---------+----------+
|label|n_reviews|sentiment|pct_of_all|
+-----+---------+---------+----------+
|    0|  1613735| negative|     25.62|
|    1|  4684444| positive|     74.38|
+-----+---------+---------+----------+



### 3.4 Review length per label (average, median, 95th percentile)
**What:** length statistics per sentiment. `percentile_approx` gives the median (50th percentile) and the 95th percentile.
**Why approximate:** an exact median needs all values sorted on one machine. `percentile_approx` builds a small summary on each partition and merges them, so it is fast on millions of rows and accurate enough for reporting. These numbers should match the EDA notebook (`02_eda_preprocessing.ipynb`, Task 2).

In [20]:
length_df = (reviews_slim.groupBy("label")
    .agg(F.count("*").alias("n_reviews"),
         F.round(F.avg("n_words"), 1).alias("avg_words"),
         F.percentile_approx("n_words", 0.5).alias("median_words"),
         F.percentile_approx("n_words", 0.95).alias("p95_words"),
         F.round(F.avg("n_chars"), 1).alias("avg_chars"),
         F.percentile_approx("n_chars", 0.5).alias("median_chars"))
    .withColumn("sentiment", F.when(F.col("label") == 1, "positive").otherwise("negative"))
    .orderBy("label"))

length_df.show()

+-----+---------+---------+------------+---------+---------+------------+---------+
|label|n_reviews|avg_words|median_words|p95_words|avg_chars|median_chars|sentiment|
+-----+---------+---------+------------+---------+---------+------------+---------+
|    0|  1613735|    134.1|          98|      368|    716.3|         523| negative|
|    1|  4684444|     91.8|          66|      251|    501.6|         360| positive|
+-----+---------+---------+------------+---------+---------+------------+---------+



### 3.5 Reviews per day of the week
**What:** number of reviews and positive share for each weekday, using the column created in 2.1.
**Why:** a quick check whether people review (or complain) more on certain days.

In [21]:
(reviews_slim.groupBy("day_of_week", "day_name")
    .agg(F.count("*").alias("n_reviews"),
         F.round(F.avg("label"), 4).alias("positive_share"))
    .orderBy("day_of_week")
    .show())

+-----------+--------+---------+--------------+
|day_of_week|day_name|n_reviews|positive_share|
+-----------+--------+---------+--------------+
|          1|     Sun|  1029995|        0.7316|
|          2|     Mon|   920412|        0.7402|
|          3|     Tue|   847839|        0.7482|
|          4|     Wed|   852328|        0.7518|
|          5|     Thu|   828894|        0.7522|
|          6|     Fri|   852923|        0.7466|
|          7|     Sat|   965788|        0.7394|
+-----------+--------+---------+--------------+



## 4. Joins: reviews + business
Reviews only have a `business_id`. To analyse sentiment by **city, state or category**, we join them with the business table on `business_id`.

### 4.1 Check the join key first (`left_anti` join)
**What:** a `left_anti` join returns the reviews whose `business_id` has **no** match in the business table.
**Why:** it tells us which join type is safe. If the result is 0, an `inner` join loses no reviews.

In [22]:
orphan_reviews = reviews_slim.join(business_clean, on="business_id", how="left_anti").count()
print(f"Reviews without a matching business: {orphan_reviews:,}")

Reviews without a matching business: 0


### 4.2 Inner join with a broadcast
**What:** join every review with its business (name, city, state, categories).
**Join type, `inner`:** we only want reviews that have business information. Since 4.1 found no orphans (expected 0), `inner` and `left` give the same rows, and `inner` states our intent more clearly.
**Why `broadcast`:** the business table is small (150K rows). `F.broadcast()` sends a full copy to every worker, so the 6.3M reviews do **not** need to be shuffled across the cluster. This is a *broadcast hash join*, the fastest join when one side is small.
**Cache:** the joined table is reused by all of Section 4, Section 7, and the comparisons in Section 5.

In [23]:
biz_small = business_clean.select("business_id", "name", "city", "state", "categories")

reviews_biz = (reviews_slim
    .join(F.broadcast(biz_small), on="business_id", how="inner")
    .cache())

print(f"Joined rows: {reviews_biz.count():,}  (reviews: {total_reviews:,})")
reviews_biz.select("name", "city", "state", "stars", "label", "year").show(5, truncate=30)

Joined rows: 6,298,179  (reviews: 6,298,179)
+-----------------+------------+-----+-----+-----+----+
|             name|        city|state|stars|label|year|
+-----------------+------------+-----+-----+-----+----+
|    Keg & Kitchen|    Westmont|   NJ|  4.0|    1|2019|
|   Tin Roof Tacos|       Boise|   ID|  4.0|    1|2019|
|    Antney's Grub|Philadelphia|   PA|  5.0|    1|2018|
|      Burger King|   Hazelwood|   MO|  2.0|    0|2021|
|Hansen's Sno-Bliz| New Orleans|   LA|  5.0|    1|2015|
+-----------------+------------+-----+-----+-----+----+
only showing top 5 rows


### 4.3 Top cities by number of reviews
**What:** reviews, positive share, average stars and number of businesses per city.
**Why:** we group by `city` **and** `state` because the same city name exists in different states.

In [24]:
city_df = (reviews_biz.groupBy("city", "state")
    .agg(F.count("*").alias("n_reviews"),
         F.round(F.avg("label"), 4).alias("positive_share"),
         F.round(F.avg("stars"), 3).alias("avg_stars"),
         F.countDistinct("business_id").alias("n_businesses"))
    .orderBy(F.desc("n_reviews")))

city_df.show(15)

+----------------+-----+---------+--------------+---------+------------+
|            city|state|n_reviews|positive_share|avg_stars|n_businesses|
+----------------+-----+---------+--------------+---------+------------+
|    Philadelphia|   PA|   853792|        0.7709|     3.89|       14567|
|     New Orleans|   LA|   567808|        0.8109|    4.057|        6208|
|           Tampa|   FL|   411866|        0.7423|    3.831|        9047|
|       Nashville|   TN|   405129|        0.7661|    3.912|        6968|
|          Tucson|   AZ|   368433|         0.722|     3.77|        9249|
|    Indianapolis|   IN|   325004|        0.7749|    3.916|        7540|
|            Reno|   NV|   321657|        0.7324|    3.818|        5932|
|   Santa Barbara|   CA|   247690|         0.805|    4.092|        3829|
|     Saint Louis|   MO|   226025|        0.7739|    3.914|        4827|
|           Boise|   ID|    96513|         0.744|    3.862|        2937|
|        Edmonton|   AB|    87402|        0.7234|  

### 4.4 States by number of reviews
**What:** the same statistics per state.

In [25]:
state_df = (reviews_biz.groupBy("state")
    .agg(F.count("*").alias("n_reviews"),
         F.round(F.avg("label"), 4).alias("positive_share"),
         F.countDistinct("business_id").alias("n_businesses"))
    .orderBy(F.desc("n_reviews")))

state_df.show(15)

+-----+---------+--------------+------------+
|state|n_reviews|positive_share|n_businesses|
+-----+---------+--------------+------------+
|   PA|  1423905|        0.7363|       34039|
|   FL|  1057396|        0.7409|       26329|
|   LA|   681270|        0.7906|        9924|
|   TN|   553658|        0.7449|       12056|
|   MO|   450405|        0.7436|       10913|
|   IN|   440783|        0.7621|       11247|
|   NV|   394764|        0.7246|        7715|
|   AZ|   392867|        0.7198|        9912|
|   CA|   319942|        0.7931|        5203|
|   NJ|   235038|        0.6837|        8536|
|   ID|   144389|        0.7327|        4467|
|   AB|    94119|        0.7231|        5572|
|   DE|    62837|        0.6741|        2265|
|   IL|    46525|        0.6695|        2145|
|   MA|       43|        0.0698|           2|
+-----+---------+--------------+------------+
only showing top 15 rows


### 4.5 Categories: `explode` the comma-separated list
**What:** `categories` is one string such as `"Restaurants, Pizza, Italian"`. We `split` it on commas, `explode` it into one row per category, and `trim` the spaces. Then we aggregate per category.
**Note:** a review of a business with 3 categories is counted once in **each** of those categories, so category counts add up to more than the number of reviews.
**Cache:** `category_df` is small (one row per category) and is reused below, in Section 5 and in Section 7.

In [26]:
review_cats = (reviews_biz
    .select("label", "stars", F.explode(F.split(F.col("categories"), ",")).alias("category"))
    .withColumn("category", F.trim(F.col("category")))
    .filter(F.col("category") != ""))

category_df = (review_cats.groupBy("category")
    .agg(F.count("*").alias("n_reviews"),
         F.round(F.avg("label"), 4).alias("positive_share"),
         F.round(F.avg("stars"), 3).alias("avg_stars"))
    .cache())

print(f"Distinct categories: {category_df.count():,}")
category_df.orderBy(F.desc("n_reviews")).show(15, truncate=False)

Distinct categories: 1,312
+-------------------------+---------+--------------+---------+
|category                 |n_reviews|positive_share|avg_stars|
+-------------------------+---------+--------------+---------+
|Restaurants              |4181250  |0.7676        |3.897    |
|Food                     |1624471  |0.7937        |3.999    |
|Nightlife                |1348878  |0.773         |3.904    |
|Bars                     |1274334  |0.7732        |3.905    |
|American (Traditional)   |881445   |0.7342        |3.767    |
|American (New)           |864147   |0.7864        |3.959    |
|Breakfast & Brunch       |769304   |0.7937        |3.991    |
|Sandwiches               |615254   |0.7835        |3.957    |
|Seafood                  |548088   |0.7839        |3.964    |
|Event Planning & Services|541920   |0.7298        |3.777    |
|Shopping                 |485036   |0.7057        |3.719    |
|Pizza                    |424277   |0.7349        |3.782    |
|Coffee & Tea             |3

### 4.6 Most negative and most positive categories (at least 50,000 reviews)
**What:** categories sorted by positive share, keeping only categories with **≥ 50,000 reviews** (and without `"Unknown"`).
**Why the minimum:** a category with 30 reviews can have an extreme share just by chance. With 50K+ reviews the share is reliable.

In [27]:
MIN_REVIEWS = 50_000
big_categories = category_df.filter((F.col("n_reviews") >= MIN_REVIEWS) & (F.col("category") != "Unknown"))

print(f"Categories with >= {MIN_REVIEWS:,} reviews: {big_categories.count()}")
print("Most NEGATIVE categories:")
big_categories.orderBy("positive_share").show(10, truncate=False)
print("Most POSITIVE categories:")
big_categories.orderBy(F.desc("positive_share")).show(10, truncate=False)

Categories with >= 50,000 reviews: 102
Most NEGATIVE categories:
+---------------------+---------+--------------+---------+
|category             |n_reviews|positive_share|avg_stars|
+---------------------+---------+--------------+---------+
|Car Dealers          |63641    |0.4773        |2.9      |
|Real Estate          |53540    |0.4904        |2.929    |
|Fast Food            |207340   |0.528         |3.032    |
|Auto Parts & Supplies|59265    |0.5361        |3.129    |
|Hotels               |165313   |0.5868        |3.236    |
|Doctors              |56447    |0.5916        |3.364    |
|Home Services        |230918   |0.6029        |3.391    |
|Automotive           |229956   |0.6032        |3.383    |
|Auto Repair          |130376   |0.6104        |3.424    |
|Oil Change Stations  |54611    |0.6202        |3.461    |
+---------------------+---------+--------------+---------+
only showing top 10 rows
Most POSITIVE categories:
+----------------------+---------+--------------+---------

## 5. Spark SQL
Spark SQL lets us query DataFrames with normal SQL. SQL and the DataFrame API are two ways of writing the **same** thing: both are turned into the same optimised plan by Spark's *Catalyst optimizer*. Below we write each query in SQL and check that it returns **exactly** the same rows as the DataFrame version from Sections 3–4.

**How we compare:** `a.exceptAll(b)` returns rows of `a` that are not in `b`. If this is empty in both directions, the two results are identical.

### 5.1 Register the DataFrames as SQL views
**What:** `createOrReplaceTempView` gives a DataFrame a table name that SQL can use. It copies no data; the view only points to the DataFrame.

In [28]:
reviews_slim.createOrReplaceTempView("reviews")
business_clean.createOrReplaceTempView("business")

def same_result(sql_df, api_df):
    # True if both DataFrames contain exactly the same rows (in any order)
    return sql_df.exceptAll(api_df).count() == 0 and api_df.exceptAll(sql_df).count() == 0

spark.sql("SHOW TABLES").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|         | business|       true|
|         |  reviews|       true|
+---------+---------+-----------+



### 5.2 Query 1: Yearly trend (`GROUP BY`)
Same as `yearly_df` in Section 3.1.

In [29]:
q1 = spark.sql('''
    SELECT year,
           COUNT(*)             AS n_reviews,
           ROUND(AVG(label), 4) AS positive_share
    FROM reviews
    GROUP BY year
    ORDER BY year
''')
q1.show(20)

check_q1 = same_result(q1, yearly_df.select("year", "n_reviews", "positive_share"))
print("Matches DataFrame API (3.1):", check_q1)

+----+---------+--------------+
|year|n_reviews|positive_share|
+----+---------+--------------+
|2005|      683|         0.877|
|2006|     3201|        0.8903|
|2007|    12458|        0.8511|
|2008|    39220|        0.8384|
|2009|    61614|        0.7952|
|2010|   115549|        0.7938|
|2011|   194809|        0.7802|
|2012|   245187|        0.7621|
|2013|   332536|        0.7552|
|2014|   461656|        0.7535|
|2015|   616159|        0.7486|
|2016|   686637|        0.7466|
|2017|   744886|        0.7461|
|2018|   829238|        0.7414|
|2019|   832821|        0.7335|
|2020|   517544|        0.7287|
|2021|   574499|        0.7059|
|2022|    29482|        0.7094|
+----+---------+--------------+

Matches DataFrame API (3.1): True


### 5.3 Query 2: Top cities (`JOIN`)
Same as `city_df` in Section 4.3. We compare all cities, then show the top 10.

In [30]:
q2 = spark.sql('''
    SELECT b.city,
           b.state,
           COUNT(*)               AS n_reviews,
           ROUND(AVG(r.label), 4) AS positive_share
    FROM reviews r
    JOIN business b ON r.business_id = b.business_id
    GROUP BY b.city, b.state
    ORDER BY n_reviews DESC
''')
q2.show(10)

check_q2 = same_result(q2, city_df.select("city", "state", "n_reviews", "positive_share"))
print("Matches DataFrame API (4.3):", check_q2)

+-------------+-----+---------+--------------+
|         city|state|n_reviews|positive_share|
+-------------+-----+---------+--------------+
| Philadelphia|   PA|   853792|        0.7709|
|  New Orleans|   LA|   567808|        0.8109|
|        Tampa|   FL|   411866|        0.7423|
|    Nashville|   TN|   405129|        0.7661|
|       Tucson|   AZ|   368433|         0.722|
| Indianapolis|   IN|   325004|        0.7749|
|         Reno|   NV|   321657|        0.7324|
|Santa Barbara|   CA|   247690|         0.805|
|  Saint Louis|   MO|   226025|        0.7739|
|        Boise|   ID|    96513|         0.744|
+-------------+-----+---------+--------------+
only showing top 10 rows
Matches DataFrame API (4.3): True


### 5.4 Query 3: Categories with the lowest positive share (`JOIN` + `LATERAL VIEW explode` + `HAVING`)
`LATERAL VIEW explode(...)` is the SQL version of `F.explode`. `HAVING` filters **groups** after aggregation (`WHERE` filters rows before). Same as `big_categories` in Section 4.6.

In [31]:
q3 = spark.sql('''
    SELECT TRIM(cat)              AS category,
           COUNT(*)               AS n_reviews,
           ROUND(AVG(r.label), 4) AS positive_share
    FROM reviews r
    JOIN business b ON r.business_id = b.business_id
    LATERAL VIEW explode(split(b.categories, ',')) c AS cat
    WHERE TRIM(cat) <> '' AND TRIM(cat) <> 'Unknown'
    GROUP BY TRIM(cat)
    HAVING COUNT(*) >= 50000
    ORDER BY positive_share ASC
''')
q3.show(10, truncate=False)

check_q3 = same_result(q3, big_categories.select("category", "n_reviews", "positive_share"))
print("Matches DataFrame API (4.6):", check_q3)

+---------------------+---------+--------------+
|category             |n_reviews|positive_share|
+---------------------+---------+--------------+
|Car Dealers          |63641    |0.4773        |
|Real Estate          |53540    |0.4904        |
|Fast Food            |207340   |0.528         |
|Auto Parts & Supplies|59265    |0.5361        |
|Hotels               |165313   |0.5868        |
|Doctors              |56447    |0.5916        |
|Home Services        |230918   |0.6029        |
|Automotive           |229956   |0.6032        |
|Auto Repair          |130376   |0.6104        |
|Oil Change Stations  |54611    |0.6202        |
+---------------------+---------+--------------+
only showing top 10 rows
Matches DataFrame API (4.6): True


### 5.5 Query 4: Top 3 cities in each state (window function `RANK`)
**What:** a *window function* computes a value for each row using a group of related rows **without** collapsing them like `GROUP BY`. `RANK() OVER (PARTITION BY state ORDER BY n_reviews DESC)` numbers the cities inside each state from the most reviewed down. We keep the top 3 cities of states with at least 100,000 reviews.
**DataFrame version:** the same thing with `Window.partitionBy(...).orderBy(...)` and `F.rank()`.

In [32]:
q4 = spark.sql('''
    WITH city_counts AS (
        SELECT b.state, b.city,
               COUNT(*)               AS n_reviews,
               ROUND(AVG(r.label), 4) AS positive_share
        FROM reviews r
        JOIN business b ON r.business_id = b.business_id
        GROUP BY b.state, b.city
    ),
    big_states AS (
        SELECT state FROM city_counts
        GROUP BY state
        HAVING SUM(n_reviews) >= 100000
    ),
    ranked AS (
        SELECT *, RANK() OVER (PARTITION BY state ORDER BY n_reviews DESC) AS city_rank
        FROM city_counts
    )
    SELECT state, city_rank, city, n_reviews, positive_share
    FROM ranked
    WHERE city_rank <= 3 AND state IN (SELECT state FROM big_states)
    ORDER BY state, city_rank
''')
q4.show(40, truncate=False)

# Same query with the DataFrame API
w = Window.partitionBy("state").orderBy(F.desc("n_reviews"))
big_states_df = state_df.filter(F.col("n_reviews") >= 100_000).select("state")
q4_api = (city_df
    .withColumn("city_rank", F.rank().over(w))
    .filter(F.col("city_rank") <= 3)
    .join(big_states_df, on="state", how="left_semi")
    .select("state", "city_rank", "city", "n_reviews", "positive_share"))

check_q4 = same_result(q4, q4_api)
print("Matches DataFrame API (Window):", check_q4)

+-----+---------+----------------+---------+--------------+
|state|city_rank|city            |n_reviews|positive_share|
+-----+---------+----------------+---------+--------------+
|AZ   |1        |Tucson          |368433   |0.722         |
|AZ   |2        |Oro Valley      |12085    |0.7153        |
|AZ   |3        |Marana          |4077     |0.6846        |
|CA   |1        |Santa Barbara   |247690   |0.805         |
|CA   |2        |Goleta          |41632    |0.7342        |
|CA   |3        |Carpinteria     |15947    |0.7719        |
|FL   |1        |Tampa           |411866   |0.7423        |
|FL   |2        |Clearwater      |80095    |0.736         |
|FL   |3        |Saint Petersburg|72269    |0.7706        |
|ID   |1        |Boise           |96513    |0.744         |
|ID   |2        |Meridian        |34130    |0.6949        |
|ID   |3        |Eagle           |8966     |0.7509        |
|IN   |1        |Indianapolis    |325004   |0.7749        |
|IN   |2        |Carmel          |37348 

### 5.6 SQL vs DataFrame API: result

In [33]:
checks = {"Q1 yearly trend": check_q1, "Q2 top cities (JOIN)": check_q2,
          "Q3 categories (JOIN + HAVING)": check_q3, "Q4 RANK per state (window)": check_q4}
for name, ok in checks.items():
    print(f"{'OK ' if ok else 'DIFF'}  {name}")
assert all(checks.values()), "A SQL query does not match its DataFrame version"
print(f"\n{sum(checks.values())} / {len(checks)} SQL queries match the DataFrame API exactly.")

OK   Q1 yearly trend
OK   Q2 top cities (JOIN)
OK   Q3 categories (JOIN + HAVING)
OK   Q4 RANK per state (window)

4 / 4 SQL queries match the DataFrame API exactly.


## 6. Text preprocessing pipeline (Spark ML): the NLP preparation
Machine learning models cannot read raw text. We turn each review into a clean list of words (**tokens**) with a Spark ML **Pipeline**: a chain of stages, where each stage reads a column and writes a new one.

| Stage | Input → Output | What it does |
|---|---|---|
| `SQLTransformer` | `text` → `clean_text` | lowercase, remove URLs, `n't` → ` not`, keep letters only, collapse spaces |
| `RegexTokenizer` | `clean_text` → `tokens` | split on spaces, drop 1-letter tokens |
| `StopWordsRemover` | `tokens` → `filtered` | remove common words (*the, is, and*), but **keep** negations and intensifiers |
| `NGram(n=2)` | `filtered` → `bigrams` | pairs of neighbouring words, e.g. `not good` |

The pipeline is fitted and checked on the **5% sample**, then saved to Drive so Phase 3 uses exactly the same preprocessing.

### 6.1 Stop words: keep negations and intensifiers
**What:** start from Spark's default English stop-word list, but remove from it the words that carry sentiment.
**Why:** *"not good"* and *"good"* mean opposite things. If we removed `not`, the model could not tell them apart. Intensifiers such as `very`, `too`, `never` also change the strength of an opinion.

In [34]:
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import SQLTransformer, RegexTokenizer, StopWordsRemover, NGram

KEEP_WORDS = {"not", "no", "nor", "never", "very", "too", "but", "against",
              "only", "most", "more", "few", "off", "over", "again"}

default_stops = StopWordsRemover.loadDefaultStopWords("english")
custom_stops  = [w for w in default_stops if w not in KEEP_WORDS]

print(f"Default English stop words : {len(default_stops)}")
print(f"Our stop-word list         : {len(custom_stops)}")
print("Kept on purpose            :", sorted(KEEP_WORDS & set(default_stops)))

Default English stop words : 181
Our stop-word list         : 167
Kept on purpose            : ['again', 'against', 'but', 'few', 'more', 'most', 'no', 'nor', 'not', 'off', 'only', 'over', 'too', 'very']


### 6.2 Cleaning stage written in Spark SQL (`SQLTransformer`)
**What:** each cleaning step is one `regexp_replace` (find a regex pattern, replace it). We list the steps in order and build one SQL expression from them. The `print` shows the final statement.

| # | Pattern | Replaced by | Why |
|---|---|---|---|
| 1 | curly apostrophes `’ ‘ ʼ` | `'` | so that `didn’t` and `didn't` are treated the same |
| 2 | `https?://...` or `www....` | space | URLs are not opinion words |
| 3–4 | `won't`, `can't` | `will not`, `can not` | irregular negations (otherwise they become `wo not` / `ca not`) |
| 5 | `n't` | ` not` | `didn't` → `did not`, so `not` survives as a word |
| 6 | anything that is not `a–z` | space | removes digits, punctuation, emojis |
| 7 | several spaces | one space | clean tokens |

**Regex escaping (important):** Spark SQL treats a backslash inside `'...'` as an escape character, just like Python. To give the regex engine `\S` we must write `\\S` in the SQL text. That is why the patterns below use **double** backslashes inside Python raw strings (`r"..."`). A single `\b` would become a *backspace* character and silently match nothing. The apostrophe is written as `\\x27` (its hex code) so it does not end the SQL string.

**Why SQL and not a Python function (UDF):** SQL expressions run inside Spark's JVM engine, which is much faster than sending every review to Python.

In [35]:
CLEANING_STEPS = [
    # (regex pattern,                    replacement as a SQL expression)
    (r"[\\u2018\\u2019\\u02bc]",           "chr(39)"),       # 1. curly apostrophes -> '
    (r"https?://\\S+|www\\.\\S+",          "' '"),           # 2. remove URLs
    (r"\\bwon\\x27t\\b",                   "'will not'"),    # 3. won't -> will not
    (r"\\bcan\\x27t\\b",                   "'can not'"),     # 4. can't -> can not
    (r"n\\x27t\\b",                        "' not'"),        # 5. didn't -> did not
    (r"[^a-z]",                            "' '"),           # 6. keep letters only
    (r" +",                                "' '"),           # 7. collapse spaces
]

expr = "lower(coalesce(text, ''))"
for pattern, replacement in CLEANING_STEPS:
    expr = f"regexp_replace({expr}, '{pattern}', {replacement})"

CLEAN_SQL = f"SELECT *, trim({expr}) AS clean_text FROM __THIS__"
print(CLEAN_SQL)

SELECT *, trim(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(lower(coalesce(text, '')), '[\\u2018\\u2019\\u02bc]', chr(39)), 'https?://\\S+|www\\.\\S+', ' '), '\\bwon\\x27t\\b', 'will not'), '\\bcan\\x27t\\b', 'can not'), 'n\\x27t\\b', ' not'), '[^a-z]', ' '), ' +', ' ')) AS clean_text FROM __THIS__


### 6.3 Build the pipeline and fit it on the sample
**What:** put the four stages in a `Pipeline` and call `fit()` on the sample. The result is a `PipelineModel`.
**Why `fit` on the sample:** these four stages have nothing to learn from data, so fitting only packages them into one reusable object. In Phase 3 we will add stages that *do* learn (e.g. `CountVectorizer`, `IDF`); keeping one pipeline object makes that easy.

In [36]:
cleaner   = SQLTransformer(statement=CLEAN_SQL)
tokenizer = RegexTokenizer(inputCol="clean_text", outputCol="tokens",
                           pattern=r"\s+", gaps=True, minTokenLength=2)
remover   = StopWordsRemover(inputCol="tokens", outputCol="filtered", stopWords=custom_stops)
bigrammer = NGram(n=2, inputCol="filtered", outputCol="bigrams")

pipeline   = Pipeline(stages=[cleaner, tokenizer, remover, bigrammer])
prep_model = pipeline.fit(sample)
print("Stages:", [type(s).__name__ for s in prep_model.stages])

Stages: ['SQLTransformer', 'RegexTokenizer', 'StopWordsRemover', 'NGram']


### 6.4 Test on handmade sentences
**What:** run the pipeline on 3 sentences we wrote ourselves, where we know the correct answer, and check it with `assert`.
**What we expect:**
1. `NOT` and `didn't` → `not` survives, and the bigram **`not good`** appears.
2. Both URLs are removed completely.
3. Curly `can’t` → `can not`, `won't` → `will not`, and `never`, `again` are kept.

In [37]:
test_df = spark.createDataFrame([
    (1, "This is NOT good. I didn't like it at all!!"),
    (2, "See the menu at https://www.yelp.com/biz/abc-123 or www.example.com - very tasty food :)"),
    (3, "The waiter can’t smile and we won't come back. Never again..."),
], ["id", "text"])

test_out = prep_model.transform(test_df)
test_out.select("id", "clean_text").show(truncate=False)
test_out.select("id", "filtered", "bigrams").show(truncate=False)

rows = {r["id"]: r for r in test_out.select("id", "clean_text", "filtered", "bigrams").collect()}  # 3 rows only
assert "not" in rows[1]["filtered"] and "not good" in rows[1]["bigrams"]
assert "did not like" in rows[1]["clean_text"]
assert not any(w in rows[2]["clean_text"] for w in ["http", "www", "yelp", "example"])
assert "very" in rows[2]["filtered"]
assert "can not" in rows[3]["clean_text"] and "will not" in rows[3]["clean_text"]
assert "never" in rows[3]["filtered"] and "again" in rows[3]["filtered"]
print("All handmade checks passed: negations kept, URLs removed, 'not good' bigram found.")

+---+--------------------------------------------------------------+
|id |clean_text                                                    |
+---+--------------------------------------------------------------+
|1  |this is not good i did not like it at all                     |
|2  |see the menu at or very tasty food                            |
|3  |the waiter can not smile and we will not come back never again|
+---+--------------------------------------------------------------+

+---+---------------------------------------------------+--------------------------------------------------------------------------------+
|id |filtered                                           |bigrams                                                                         |
+---+---------------------------------------------------+--------------------------------------------------------------------------------+
|1  |[not, good, not, like]                             |[not good, good not, not like]            

### 6.5 Token statistics on the sample
**What:** apply the pipeline to the 5% sample and measure average tokens per review before and after stop-word removal, and the vocabulary size (number of distinct words).
**Why:** it shows how much the text shrinks, and the vocabulary size helps choose the feature size for Phase 3 (e.g. `CountVectorizer(vocabSize=...)`).
**Cache:** `sample_prep` is reused by three statistics, so we cache it and free it afterwards.

In [38]:
sample_prep = (prep_model.transform(sample)
               .select("review_id", "label", "clean_text", "tokens", "filtered", "bigrams")
               .cache())

sample_prep.select("label", "clean_text", "filtered").show(3, truncate=60)

(sample_prep.agg(
    F.count("*").alias("reviews"),
    F.round(F.avg(F.size("tokens")), 1).alias("avg_tokens_before"),
    F.round(F.avg(F.size("filtered")), 1).alias("avg_tokens_after"),
    F.round(1 - F.sum(F.size("filtered")) / F.sum(F.size("tokens")), 3).alias("share_removed"))
 .show())

vocab_before = sample_prep.select(F.explode("tokens").alias("w")).distinct().count()
vocab_after  = sample_prep.select(F.explode("filtered").alias("w")).distinct().count()
print(f"Vocabulary size before stop words: {vocab_before:,}")
print(f"Vocabulary size after  stop words: {vocab_after:,}")

+-----+------------------------------------------------------------+------------------------------------------------------------+
|label|                                                  clean_text|                                                    filtered|
+-----+------------------------------------------------------------+------------------------------------------------------------+
|    1|i live and work about ish minutes away from here but i wi...|[live, work, ish, minutes, away, but, happily, willing, d...|
|    1|not our st time here as always good food good service and...|[not, st, time, always, good, food, good, service, fun, a...|
|    1|had the curry chicken it was very good very flavorful and...|[curry, chicken, very, good, very, flavorful, service, li...|
+-----+------------------------------------------------------------+------------------------------------------------------------+
only showing top 3 rows
+-------+-----------------+----------------+-------------+
|review

### 6.6 Save the fitted pipeline to Drive
**What:** save the `PipelineModel` (overwrite if it exists), load it back and run it once to prove the saved copy works.
**Why:** Phase 3 loads it with `PipelineModel.load(...)`, so the model is trained on text cleaned **exactly** the same way.

In [39]:
prep_model.write().overwrite().save(PIPELINE_PATH)

loaded_model = PipelineModel.load(PIPELINE_PATH)
loaded_model.transform(test_df).select("id", "filtered").show(truncate=False)
print("Saved to:", PIPELINE_PATH)

# free the memory used by the sample
sample_prep.unpersist()
sample.unpersist()

+---+---------------------------------------------------+
|id |filtered                                           |
+---+---------------------------------------------------+
|1  |[not, good, not, like]                             |
|2  |[see, menu, very, tasty, food]                     |
|3  |[waiter, not, smile, not, come, back, never, again]|
+---+---------------------------------------------------+

Saved to: /content/drive/MyDrive/yelp_project/models/preprocessing_pipeline


DataFrame[review_id: string, business_id: string, stars: double, text: string, date: date, label: int, year: int]

## 7. Export small tables for the dashboard (Phase 4)
**What:** save four **small aggregated tables** (computed by Spark above on the full data) as Parquet in `dashboard_data/`:

| Table | Rows (approx.) | Content |
|---|---|---|
| `yearly_stats` | 18 | reviews, positive share, avg stars, avg words per year |
| `city_stats` | 50 | top 50 cities by review count |
| `category_stats` | ~1,300 | every category with review count and positive share |
| `length_stats` | 2 | length statistics per sentiment |

**Why:** the Streamlit dashboard should open instantly and cannot run Spark on 6.3M reviews for every click. Spark does the heavy work **here**; the dashboard only reads these small results (e.g. `pandas.read_parquet`). Spark is still the engine that produced every number.
`coalesce(1)` writes each table as a single file, which is easier for the dashboard to read.

In [40]:
os.makedirs(DASHBOARD_PATH, exist_ok=True)

exports = {
    "yearly_stats":   yearly_df,
    "city_stats":     city_df.limit(50),
    "category_stats": category_df.orderBy(F.desc("n_reviews")),
    "length_stats":   length_df,
}

for name, table in exports.items():
    path = f"{DASHBOARD_PATH}/{name}"
    table.coalesce(1).write.mode("overwrite").parquet(path)
    print(f"{name:15s} {spark.read.parquet(path).count():>6,} rows  ->  {path}")

yearly_stats        18 rows  ->  /content/drive/MyDrive/yelp_project/dashboard_data/yearly_stats
city_stats          50 rows  ->  /content/drive/MyDrive/yelp_project/dashboard_data/city_stats
category_stats   1,312 rows  ->  /content/drive/MyDrive/yelp_project/dashboard_data/category_stats
length_stats         2 rows  ->  /content/drive/MyDrive/yelp_project/dashboard_data/length_stats


### 7.1 Free memory
**What:** remove the cached DataFrames from memory now that we are done with them.

In [41]:
for cached in [category_df, reviews_biz, reviews_slim]:
    cached.unpersist()
print("Cached DataFrames released.")

Cached DataFrames released.


## 8. Phase 2 summary
*(Numbers from the run on Colab.)*

| Metric | Value |
|---|---|
| Reviews / businesses loaded | 6,298,179 / 150,346 |
| Null values in reviews | 0 (all 7 columns) |
| Businesses with missing categories → `"Unknown"` | 103 |
| Duplicate `review_id` / `business_id` | 0 / 0 |
| Reviews without a matching business (`left_anti`) | 0 |
| Reviews with > 500 words (positive share) | 56,681 (41.6% positive; all reviews: 74.4%) |
| Avg / median / p95 words: negative | 134.1 / 98 / 368 |
| Avg / median / p95 words: positive | 91.8 / 66 / 251 |
| Top city by reviews | Philadelphia, PA: 853,792 reviews (77.09% positive) |
| Distinct categories | 1,312 |
| Most negative category (≥ 50K reviews) | Car Dealers (47.73% positive, 63,641 reviews) |
| SQL queries matching the DataFrame API | 4 / 4 |
| Sample size (5%) | 314,338 |
| Avg tokens per review: before → after stop words | 97.3 → 54.6 (43.9% of tokens removed) |
| Vocabulary size (after stop words, sample) | 118,665 (118,778 before) |

**Spark features demonstrated**

| Feature | Where |
|---|---|
| DataFrames, schemas, Parquet I/O | 0.4, 1.1, 6.6, 7 |
| Missing-value handling (nulls, empty strings, fill with `"Unknown"`) | 1.2 – 1.6 |
| Transformations: `withColumn`, `select` / `drop`, `filter`, `when`, `between` | 2.1 – 2.3 |
| Aggregations: `groupBy` / `agg`, `countDistinct`, `percentile_approx` | 3.1 – 3.5, 4.3 – 4.6 |
| Joins: `left_anti` (key check), `inner` with `broadcast`, `left_semi` | 4.1, 4.2, 5.5 |
| `explode` / `split` on arrays | 4.5, 5.4 |
| Spark SQL: temp views, `JOIN`, `HAVING`, `LATERAL VIEW`, CTEs | 5.1 – 5.5 |
| Window functions: `RANK() OVER (PARTITION BY ...)`, `Window` | 5.5 |
| Spark ML Pipeline: `SQLTransformer`, `RegexTokenizer`, `StopWordsRemover`, `NGram` | 6 |
| Caching / unpersist, sampling | 0.5, 2.2, 4.2, 4.5, 6.5, 7.1 |

**Next (Phase 3):** load `reviews_clean.parquet` and the saved `preprocessing_pipeline`, add TF-IDF features, and train classifiers with class weights (evaluate with F1 / Recall).